### Structured Output


### Pydantic

In [7]:
import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv('OPENAI_API_KEY')
os.environ["GEMINI_API_KEY"]=os.getenv('GEMINI_API_KEY')
os.environ["GROQ_API_KEY"]=os.getenv('GROQ_API')

In [8]:
model = init_chat_model(model="groq:qwen/qwen3-32b")
model 

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A525DD8590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A525DD92B0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [9]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int= Field(description="Year in which movie was release")
    director:str=Field(description="The director of movie")
    rating:float= Field(description="The rating of movie")

In [10]:
model_with_structured_output =model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A525DD8590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A525DD92B0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [11]:
model_with_structured_output.invoke("provide information about movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [12]:
class MovieWthOptionlData(BaseModel):
    title:str = Field(..., description="The title of the movie")
    year:int= Field(..., description="Year in which movie was release")
    director:str=Field(...,description="The director of movie")
    rating:float= Field(..., description="The rating of movie")

In [13]:
model_with_structured_output_2 = model.with_structured_output(MovieWthOptionlData, include_raw=True)
model_with_structured_output_2.invoke("provide information about movie Inception", )


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for information about the movie Inception. Let me see what I need to do here. The available tool is the MovieWthOptionlData function, which requires title, year, director, and rating. I need to make sure I have all these details.\n\nFirst, the title is Inception. I know that's correct. The year it was released... I think it's 2010. Let me confirm that. Yeah, Christopher Nolan directed it, right? And the rating... I remember it has a high rating on IMDb, maybe around 8.8. Wait, I should double-check these details to be accurate. \n\nIf I get the year wrong, that could be an issue. Let me think. Nolan directed Inception, which came out in 2010. Yes, that's right. The rating is 8.8/10. Okay, so all the required parameters are there. I need to structure the function call with these parameters. Make sure the JSON is correctly formatted with the parameters in the required order. No typos in the di

### Nested structure

In [14]:
class Actor(BaseModel):
    name:str=Field(description="Name of actor")
    role:str=Field(description="Role of actor")
class MovieDetails(BaseModel):
    title:str = Field(..., description="The title of the movie")
    year:int= Field(..., description="Year in which movie was release")
    director:str=Field(...,description="The director of movie")
    rating:float= Field(..., description="The rating of movie"), 
    cast:list[Actor]=Field(..., description="The cast of the movie")
    budget:float | None = Field(..., description="Budget in million USD")
        


In [15]:
model_with_structured_output_2 = model.with_structured_output(MovieDetails)
model_with_structured_output_2.invoke("provide information about movie Inception", )

e:\gen_ai\agentic ai\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The rating of movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


MovieDetails(title='Inception', year=2010, director='Christopher Nolan', rating=(FieldInfo(annotation=NoneType, required=True, description='The rating of movie'),), cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], budget=160.0)

### TypeDict

In [16]:
from typing_extensions import TypedDict,Annotated
class Movie(TypedDict):
    """A movie with detail."""
    title: Annotated[str , ...,"The title of the movie"]
    year: Annotated[int, ...,"Year in which movie was release"]
    director:Annotated[str,..., "The director of movie"]
    rating: Annotated[float,...,"The rating of movie"]



In [18]:
model_with_structured_output3 =model.with_structured_output(Movie)
model_with_structured_output3.invoke("provide information about movie Kick")

{'director': 'S. S. Rajamouli', 'rating': 8.5, 'title': 'Kick', 'year': 2014}

In [22]:
class Actor(TypedDict):
    name:str
    role:str
class MovieDetails1(TypedDict):
    title:str 
    year:int
    director:str
    rating:float
    cast:list[Actor]
    budget:float | None = Field(..., description="Budget in million USD")
        


In [26]:
model_with_structured_output4 =model.with_structured_output(MovieDetails1)
model_with_structured_output4.invoke("provide information about movie Kick")

{'budget': 45000000,
 'cast': [{'name': 'Vijay Devarakonda', 'role': 'Nani'},
  {'name': 'Megha Akash', 'role': 'Megha'},
  {'name': 'Sumanth', 'role': 'Surya'},
  {'name': 'Prakash Raj', 'role': 'Father'},
  {'name': 'Anjali Devi', 'role': 'Mother'}],
 'director': 'Shiva Nirupula',
 'rating': 7.3,
 'title': 'Kick',
 'year': 2014}

### DataClass

In [31]:
from langchain.agents import create_agent
from dataclasses import dataclass
@dataclass
class ContactInfo:
    """Contact info of a person"""
    name: str
    email:str
    phoneNumber:str


agent =create_agent(model="gpt-4.1",response_format=ContactInfo )     


In [32]:
agent.invoke({"role":"user", "content":"get the contact info  of user subhan , subhan@gmail.com, +122222223"})

BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'messages': empty array. Expected an array with minimum length 1, but got an empty array instead.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'empty_array'}}